# 11 — The console with the fine-tuned reading, hosted from Colab

**What this does:** runs the review console (`demo/app.py`) on a Colab GPU with the approved fine-tuned model
switched on for the four demo images only (`demo/lab_demo.py`), and gives you a public link to it. Nothing is
published and the weights stay in your Drive.

**Two parts:**
- **A. Check run and recorded fallback (about 10 min):** the four demo images go through the console's own code
  once, with the anomaly check's reference files present this time. It prints what the console will show for each
  and saves a recorded copy of every result page to Drive, to open if the live link fails during the call.
- **B. Launch (on the call day, about 15 min before):** run all cells again; part A doubles as a warm-up. The last
  cell prints the link. Keep the tab open: the link dies with the session.

**Before running:** upload `lab_demo_bundle.zip` (about 94 MB, it carries the anomaly reference files) to
`MyDrive/cultureqc/staged/`. The approved fine-tuned weights are already there (`mcellseg_ftF_r2.weights`).

**I/O rule.** Nothing is read through the Drive mount except single copies to local disk (the bundle and the
weights); the recorded pages go back as one zip.

**Runtime:** any GPU works; A100 or L4 if offered.

**Bring back after part A:** the table printed by the check run (paste it to me), or
`MyDrive/cultureqc/staged/lab_demo_recorded.zip`.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, shutil, time, subprocess, json, hashlib, zipfile
DRIVE_ROOT = '/content/drive/MyDrive/cultureqc'
STAGE_DIR = f'{DRIVE_ROOT}/staged'
BUNDLE = f'{STAGE_DIR}/lab_demo_bundle.zip'
WEIGHTS = f'{STAGE_DIR}/mcellseg_ftF_r2.weights'
RECORDED_ZIP = f'{STAGE_DIR}/lab_demo_recorded.zip'
EXPECTED_COMMIT = '643e557'        # the commit the bundle was made from
REPO = '/content/cultureqc'
assert os.path.exists(BUNDLE), f'upload the bundle first: {BUNDLE}'
assert os.path.exists(WEIGHTS), f'the approved fine-tuned weights are missing: {WEIGHTS}'
print(f'{BUNDLE}: {os.path.getsize(BUNDLE)/1e6:.1f} MB; weights {os.path.getsize(WEIGHTS)/1e9:.2f} GB')

In [ ]:
t0 = time.time()
shutil.rmtree(REPO, ignore_errors=True)
shutil.copy(BUNDLE, '/content/bundle.zip')                 # one sequential read from Drive
!unzip -q -o /content/bundle.zip -d $REPO
os.remove('/content/bundle.zip')
%cd $REPO
sys.path.insert(0, REPO)
commit = open('COMMIT').read().strip()
print('bundle commit', commit, f'— unpacked in {time.time()-t0:.0f} s')
assert commit.startswith(EXPECTED_COMMIT), (commit, EXPECTED_COMMIT)

In [ ]:
!pip install -q gradio==6.26.0 cellpose==4.2.1.1 timm grad-cam transformers huggingface_hub jsonschema plotly pandas matplotlib pyyaml scikit-image opencv-python-headless tifffile
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Runtime > Change runtime type > GPU'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# The shipped Cellpose-SAM weights Colab downloads must be the approved ones (configs/approved_changes.jsonl)
from cellpose import models
_ = models.CellposeModel(gpu=True)
w = os.path.join(str(models.MODEL_DIR), 'cpsam_v2')
h = hashlib.sha256(open(w, 'rb').read()).hexdigest()
print('cpsam_v2', h)
assert h == '0f1cc3f7ecdd8a037a57c6c48d9d8921391be4cbce3fa9f13c3e3a2e1253c667', 'not the shipped weights: stop and tell me'
# The approved fine-tuned weights: one copy from Drive to local disk, where the product looks for them
t0 = time.time()
dst = 'cache/finetune/mcellseg_ftF_r2/models/mcellseg_ftF_r2'
os.makedirs(os.path.dirname(dst), exist_ok=True)
shutil.copy(WEIGHTS, dst)
h = hashlib.sha256(open(dst, 'rb').read()).hexdigest()
print('mcellseg_ftF_r2', h, f'— copied in {time.time()-t0:.0f} s')
assert h == '2d1549682c1da4829e7766bebf6e90e2373aa6c439f5138b098a71259d1239a2', 'not the approved fine-tuned weights: stop and tell me'
# The files that decide what the console shows are byte-identical to the commit
SEALED = {
 "demo/lab_demo.py": "e18c91bbb8631bc66e9dcad415933f915563d1144963a48109bf19eaf749fcee",
 "demo/app.py": "96b85e130b169d6119537f3c711bcc3588883955f267f9bc8b2188f4dc5ae553",
 "demo/analysis.py": "7dcfa0e146b50b335f03a6410817653d5c7467e4d3037f2672931d5bc9d13918",
 "scripts/lab_demo_record.py": "e1def5183c7bdd39e84bb70e98767383dc851e0f24e5030f5d292f1859768331",
 "configs/approved_changes.jsonl": "455fcc1f65583155dc8831cacbf1936648a549a0bc56c78d1c5a97982b88e881",
 "configs/finetuned_models.yaml": "53e2687b0d65f6099dc5ef95e70a7b2013647c9f5fe82a9c2462e076f4bb9a52",
 "culture/finetuned.py": "d99a4be893b835372b48c607d90bfa2b525c960b81f16d541dcc2ae925d90c70",
 "culture/approvals.py": "16710f519e51274758b28ec450cba8ef455743f717392cd765ff74b1cc169b3b",
 "culture/seg.py": "045f0499c3e90e7f99fc180f64dfebe256fed08bb9da1e9ac3a2f4156faf5fe6",
 "culture/rules.py": "7ab253633f1558220d043ec4517ea5fa9981fd0b820b2290d24aef824b106c89",
 "configs/anomaly.yaml": "2776ce48025a554c2d3487c0d98433ef376ee4b9de1f5b25d48508bf82721184",
 "results/finetuned_product_readings.jsonl": "e26ebfa855ab40e9f81586e75c679dd1397da36b9b7f203757d088ee63e16837"
}
for f, s in SEALED.items():
    assert hashlib.sha256(open(f, 'rb').read()).hexdigest() == s, f
print('committed files match', EXPECTED_COMMIT)
# The anomaly check's reference files must match configs/anomaly.yaml, or it shows "unavailable"
from culture.anomaly import load_anomaly_config, load_banks
banks, why = load_banks(load_anomaly_config())
assert banks is not None, why
print('anomaly banks match configs/anomaly.yaml')

In [ ]:
# mCellSeg (Alam et al. 2026, CC BY 4.0) straight from Zenodo to local disk, MD5 from the record
t0 = time.time()
os.makedirs('/content/dl', exist_ok=True)
!curl -sL -o /content/dl/mCellSeg.zip "https://zenodo.org/api/records/20174259/files/mCellSeg.zip/content"
md5 = hashlib.md5(open('/content/dl/mCellSeg.zip', 'rb').read()).hexdigest()
print('md5', md5)
assert md5 == 'ed06d6e4c10e93b81703984cef852246'
os.makedirs('data/sources/mcellseg', exist_ok=True)
!unzip -q -o /content/dl/mCellSeg.zip -d data/sources/mcellseg
os.remove('/content/dl/mCellSeg.zip')
print(f'{time.time()-t0:.0f} s')
# The product's own weights check, as it runs before each reading
!python scripts/approve_change.py check

### ⏸ Checkpoint
Above, you should see **`committed files match`**, **`anomaly banks match configs/anomaly.yaml`**,
**`cpsam_v2: match`** and **`mcellseg_ftF_r2: match`**. If any is missing, stop and send me the output.

## A. Check run and recorded fallback

First the console's stored examples are re-read live and compared with what they stored (the console's own
self-check). Then the four demo images go through the console's path once, and a table prints what the console will
show for each.

In [ ]:
!python -m demo.selfcheck --n 5

In [ ]:
shutil.rmtree('cache/lab_demo_recorded', ignore_errors=True)
r = subprocess.run('CULTUREQC_FINETUNED=mcellseg_ftF_r2 python scripts/lab_demo_record.py 2>&1 | grep -v -E "it/s]|s/it]"',
                   shell=True)
assert r.returncode == 0, 'the check run failed: send me the output above'
local = '/content/lab_demo_recorded.zip'
with zipfile.ZipFile(local, 'w', zipfile.ZIP_DEFLATED) as z:
    z.writestr('COMMIT', commit)
    for f in sorted(os.listdir('cache/lab_demo_recorded')):
        z.write(os.path.join('cache/lab_demo_recorded', f), f)
shutil.copy(local, RECORDED_ZIP)
print('recorded pages ->', RECORDED_ZIP, f'{os.path.getsize(local)/1e6:.1f} MB')

### ⏸ Checkpoint
Paste me the table above (or bring back `lab_demo_recorded.zip`). Before the call I need to see what the anomaly
check says about these four images: this is the first time it has run on them.

## B. Launch the console

The public link is printed below. Only the four demo images (the buttons under the examples) get the fine-tuned
reading; any upload gets the shipped reading only.

In [ ]:
os.environ['CULTUREQC_FINETUNED'] = 'mcellseg_ftF_r2'
from demo import app
assert len(app.LAB_IMAGES) == 4, 'the demo images are missing'
app.demo.launch(share=True, theme=app.CultureQCTheme(), css=app.CSS, head=app.viz3d.HEAD, footer_links=[])